In [2]:
import pandas as pd
pd.set_option('display.max_columns', None)

core = pd.read_csv('data/processed/hospital_admissions_core.csv', parse_dates=['admission_date', 'discharge_date'])
core.shape

(45000, 36)

In [3]:
total_admissions = core['admission_id'].nunique()
by_department = core.groupby('department_name')['admission_id'].count().sort_values(ascending=False)
print(f"Total admissions: {total_admissions}")
print(by_department)

Total admissions: 45000
department_name
Surgery              10126
Emergency             8777
Pediatrics            8438
Internal Medicine     7695
Orthopedics           5924
ICU                   4040
Name: admission_id, dtype: int64


In [4]:
avg_los_overall = core['length_of_stay'].mean()
avg_los_by_dept = core.groupby('department_name')['length_of_stay'].mean().round(1).sort_values(ascending=False)
print(f"Overall avg LOS: {avg_los_overall:.1f} days")
print(avg_los_by_dept)

Overall avg LOS: 5.2 days
department_name
ICU                  10.0
Emergency             4.7
Internal Medicine     4.7
Orthopedics           4.7
Pediatrics            4.7
Surgery               4.7
Name: length_of_stay, dtype: float64


In [5]:
core_sorted = core.sort_values(['patient_id', 'admission_date'])
core_sorted['prev_discharge'] = core_sorted.groupby('patient_id')['discharge_date'].shift(1)
core_sorted['days_since_prev_discharge'] = (core_sorted['admission_date'] - core_sorted['prev_discharge']).dt.days
core_sorted['is_readmission'] = core_sorted['days_since_prev_discharge'] <= 30

readmission_rate = core_sorted['is_readmission'].sum() / core_sorted['admission_id'].nunique() * 100
print(f"Readmission rate: {readmission_rate:.2f}%")

# merge the flag back into core
core = core.merge(core_sorted[['admission_id', 'is_readmission']], on='admission_id', how='left')
core['is_readmission'] = core['is_readmission'].fillna(False)


Readmission rate: 2.49%


In [12]:
ward = pd.read_csv('data/raw/ward.csv')
department = pd.read_csv('data/raw/department.csv')

# correct: sum total_beds per department directly from ward table
dept_bed_count = ward.groupby('department_id')['total_beds'].sum().reset_index()
dept_bed_count = dept_bed_count.merge(department[['department_id', 'department_name']], on='department_id')
print(dept_bed_count)

   department_id  total_beds    department_name
0              1          75          Emergency
1              2          65  Internal Medicine
2              3          90            Surgery
3              4          70         Pediatrics
4              5          50        Orthopedics
5              6          65                ICU


In [16]:
ward = pd.read_csv('data/raw/ward.csv')
department = pd.read_csv('data/raw/department.csv')

# bed -> ward -> department (correct join path)
occ_by_dept = bed.merge(ward[['ward_id', 'department_id']], on='ward_id', how='left')
occ_by_dept = occ_by_dept.merge(department[['department_id', 'department_name']], on='department_id', how='left')

occ_by_dept_rate = occ_by_dept.groupby('department_name')['bed_status'].apply(lambda x: (x == 'Occupied').mean() * 100).round(1)
print(occ_by_dept_rate)

department_name
Emergency            62.7
ICU                  80.0
Internal Medicine    61.5
Orthopedics          62.0
Pediatrics           61.4
Surgery              63.3
Name: bed_status, dtype: float64


In [8]:
core['los_capped'] = core['length_of_stay'].clip(lower=0)
bed_days_used = core.groupby('department_name')['los_capped'].sum()

date_range_days = (core['admission_date'].max() - core['admission_date'].min()).days
beds_per_dept = core.groupby('department_name')['total_beds'].first()  # or sum of ward-level beds
bed_days_available = beds_per_dept * date_range_days

bed_utilization = (bed_days_used / bed_days_available * 100).round(1)
print(bed_utilization)

department_name
Emergency            125.3
ICU                   92.0
Internal Medicine    163.5
Orthopedics           84.3
Pediatrics           180.7
Surgery              108.0
dtype: float64


In [9]:
dept_summary = core.groupby('department_name').agg(
    admissions=('admission_id', 'count'),
    avg_los=('length_of_stay', 'mean'),
    readmission_rate=('is_readmission', 'mean'),
    revenue=('total_amount', 'sum')
).reset_index()

# normalize each metric 0-1, then blend (lower LOS/readmission = better, so invert those)
dept_summary['los_score'] = 1 - (dept_summary['avg_los'] - dept_summary['avg_los'].min()) / (dept_summary['avg_los'].max() - dept_summary['avg_los'].min())
dept_summary['readmit_score'] = 1 - (dept_summary['readmission_rate'] - dept_summary['readmission_rate'].min()) / (dept_summary['readmission_rate'].max() - dept_summary['readmission_rate'].min())
dept_summary['volume_score'] = (dept_summary['admissions'] - dept_summary['admissions'].min()) / (dept_summary['admissions'].max() - dept_summary['admissions'].min())

# weighted blend — document these weights in your data dictionary
dept_summary['efficiency_score'] = (
    dept_summary['los_score'] * 0.4 +
    dept_summary['readmit_score'] * 0.4 +
    dept_summary['volume_score'] * 0.2
).round(3)

dept_summary.sort_values('efficiency_score', ascending=False)

,department_name,admissions,avg_los,readmission_rate,revenue,los_score,readmit_score,volume_score,efficiency_score
3,Orthopedics,5924,4.677920,0.021438,222585716,0.995910,1.000000,0.309563,0.860
0,Emergency,8777,4.692492,0.024040,329738229,0.993173,0.550586,0.778344,0.773
4,Pediatrics,8438,4.691752,0.024887,315714822,0.993312,0.404233,0.722642,0.704
5,Surgery,10126,4.674798,0.026170,377216234,0.996496,0.182653,1.000000,0.672
2,Internal Medicine,7695,4.656140,0.025601,289566059,1.000000,0.280971,0.600559,0.633
1,ICU,4040,9.980693,0.027228,149425049,0.000000,0.000000,0.000000,0.000


In [19]:
core.to_excel('data/processed/hospital_final_dataset.xlsx', index=False, sheet_name='admissions_core')

with pd.ExcelWriter('data/processed/hospital_kpi_summary.xlsx') as writer:
    dept_summary.to_excel(writer, sheet_name='department_kpis', index=False)
    bed_utilization.to_excel(writer, sheet_name='bed_utilization')
    occ_by_dept_rate.to_excel(writer, sheet_name='occupancy_by_dept')